# **Introduction to CUDA Programming with Google Colab**

**Learning Objectives**

After completing this activity, students should be able to:

1. Explain the roles of the Host (CPU) and Device (GPU).
2. Understand CUDA thread, block, and grid hierarchy.
3. Implement a simple CUDA kernel.
4. Transfer data between CPU and GPU memory.
5. Measure the impact of memory transfer overhead.
6. Compare CPU and GPU performance.

# Part A: Run the Notebook and Observe

## **A1. Enable GPU Runtime**

In Colab:
```
Runtime → Change Runtime Type → GPU
```


## **A2. Verify GPU**

In [ ]:
!nvidia-smi

### **Questions**
Record:
* GPU Model:
* Total GPU Memory:
* CUDA Version:

## **A3. Import Libraries**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time

from numba import cuda

## **A4. Create Test Image**

In [ ]:
IMAGE_SIZE = 1024

image = np.random.randint(
    0,
    256,
    (IMAGE_SIZE, IMAGE_SIZE, 3),
    dtype=np.uint8
)

print("Image Shape:", image.shape)

plt.figure(figsize=(6,6))
plt.imshow(image)
plt.title("Original Image")
plt.axis("off")
plt.show()

## **A5. CPU Brightness Function**

In [ ]:
def brightness_cpu(image, brightness):

    output = np.clip(
        image.astype(np.int32) + brightness,
        0,
        255
    )

    return output.astype(np.uint8)

## **A6. Benchmark CPU**

In [ ]:
brightness_value = 50

start = time.perf_counter()

cpu_result = brightness_cpu(
    image,
    brightness_value
)

cpu_time = time.perf_counter() - start

print(f"CPU Time: {cpu_time:.6f} seconds")

# **Part B: Complete the CUDA Implementation**

## **B1. Prediction Questions**
Before writing any code, answer:

### **Question 1:**
For a 1024 x 1024 image using blocks of 16 x 16:
* How many blocks will be launched?
* How many threads will be launched?


### **Question 2:**
Which do you predict will take the most time?
* Host → Device transfer
* CUDA Kernel
* Device → Host transfer
Why?




## **B2. Complete the CUDA Kernel**
Replace all TODO sections.

In [ ]:
@cuda.jit
def brightness_kernel(input_image, output_image, brightness):

    # TODO 1
    # Obtain thread coordinates

    row, col = _______________________

    # TODO 2
    # Check image boundary

    if __________________________________:

        for channel in range(3):

            # TODO 3
            # Increase brightness

            pixel = _______________________

            if pixel > 255:
                pixel = 255

            output_image[row, col, channel] = pixel

## **B3. Configure CUDA Grid**
Complete the missing code.

In [ ]:
threads_per_block = (16,16)

# TODO 4

blocks_per_grid_x = (
    ___________________________________
)

blocks_per_grid_y = (
    ___________________________________
)

blocks_per_grid = (
    blocks_per_grid_x,
    blocks_per_grid_y
)

print("Blocks:", blocks_per_grid)
print("Threads:", threads_per_block)

## **B4. Check Your Work**
Expected output:
```
Blocks: (64, 64)
Threads: (16, 16)
```
for a 1024 x 1024 image.

# **Part C: Benchmark CPU and GPU**

## **C1. Host → Device Transfer Time**

In [ ]:
start = time.perf_counter()

d_input = cuda.to_device(image)

cuda.synchronize()

h2d_time = time.perf_counter() - start

print(f"Host → Device: {h2d_time:.6f} seconds")


## **C2. Allocate Device Memory**

In [ ]:
d_output = cuda.device_array_like(image)

## **C3. Kernel Execution Time**

In [ ]:
cuda.synchronize()

start = time.perf_counter()

brightness_kernel[
    blocks_per_grid,
    threads_per_block
](
    d_input,
    d_output,
    brightness_value
)

cuda.synchronize()

kernel_time = time.perf_counter() - start

print(f"Kernel Time: {kernel_time:.6f} seconds")

## **C4. Device → Host Transfer Time**

In [ ]:
cuda.synchronize()

start = time.perf_counter()

gpu_result = d_output.copy_to_host()

cuda.synchronize()

d2h_time = time.perf_counter() - start

print(f"Device → Host: {d2h_time:.6f} seconds")

## **C5. Total GPU Processing Time**

Complete the calculation.

In [ ]:
total_gpu_time = (
    h2d_time +
    kernel_time +
    d2h_time
)

speedup = cpu_time / total_gpu_time

print()
print("===== PERFORMANCE SUMMARY =====")
print(f"CPU Time               : {cpu_time:.6f} s")
print(f"Host → Device Transfer : {h2d_time:.6f} s")
print(f"Kernel Execution       : {kernel_time:.6f} s")
print(f"Device → Host Transfer : {d2h_time:.6f} s")
print(f"Total GPU Time         : {total_gpu_time:.6f} s")
print(f"Overall Speedup        : {speedup:.2f}x")

## **C6. Display Result**

In [ ]:
fig, ax = plt.subplots(
    1,
    2,
    figsize=(12,6)
)

ax[0].imshow(image)
ax[0].set_title("Original")

ax[1].imshow(gpu_result)
ax[1].set_title("Brightness +50")

for a in ax:
    a.axis("off")

plt.show()

## **C7. Multi-Size Benchmark**

In [ ]:
sizes = [
    512,
    1024,
    2048,
    4096
]

results = []

for size in sizes:

    print(f"\nTesting {size} x {size}")

    image = np.random.randint(
        0,
        256,
        (size, size, 3),
        dtype=np.uint8
    )

    # CPU
    start = time.perf_counter()

    brightness_cpu(
        image,
        50
    )

    cpu_t = time.perf_counter() - start

    # H2D

    start = time.perf_counter()

    d_input = cuda.to_device(image)

    cuda.synchronize()

    h2d_t = time.perf_counter() - start

    d_output = cuda.device_array_like(image)

    # Grid

    blocks_x = (
        (image.shape[0] + 15) // 16
    )

    blocks_y = (
        (image.shape[1] + 15) // 16
    )

    # Kernel

    cuda.synchronize()

    start = time.perf_counter()

    brightness_kernel[
        (blocks_x, blocks_y),
        (16,16)
    ](
        d_input,
        d_output,
        50
    )

    cuda.synchronize()

    kernel_t = time.perf_counter() - start

    # D2H

    cuda.synchronize()

    start = time.perf_counter()

    d_output.copy_to_host()

    cuda.synchronize()

    d2h_t = time.perf_counter() - start

    total_gpu_t = (
        h2d_t +
        kernel_t +
        d2h_t
    )

    speedup = cpu_t / total_gpu_t

    results.append(
        [
            size,
            cpu_t,
            h2d_t,
            kernel_t,
            d2h_t,
            total_gpu_t,
            speedup
        ]
    )

## **C8. Display Benchmark Table**

In [ ]:
print(
    f"{'Size':<10}"
    f"{'CPU(ms)':<12}"
    f"{'H2D(ms)':<12}"
    f"{'Kernel(ms)':<14}"
    f"{'D2H(ms)':<12}"
    f"{'GPU(ms)':<12}"
    f"{'Speedup':<10}"
)

print("-"*82)

for row in results:

    print(
        f"{row[0]:<10}"
        f"{row[1]*1000:<12.3f}"
        f"{row[2]*1000:<12.3f}"
        f"{row[3]*1000:<14.3f}"
        f"{row[4]*1000:<12.3f}"
        f"{row[5]*1000:<12.3f}"
        f"{row[6]:<10.2f}"
    )

# **Part D: Reflection Questions**
Answer the following questions.

1. What is the role of the host and device in CUDA?

2. How are threads organised into blocks and grids?

3. Why is brightness adjustment considered a
   data-parallel problem?

4. Why does GPU acceleration generally improve
   as image size increases?

5. What overhead is introduced by CPU-GPU
   memory transfers?

6. How many threads are launched when using:
```
      Image = 1024 x 1024
      Block Size = 16 x 16
```
7. If the GPU contained thousands of CUDA cores,
   how can they process millions of image pixels?